# verify.ipynb — Cell-by-cell debugging

This notebook is for **manually verifying** the new pipeline against the legacy
results, table by table. Run cells top-to-bottom.

Sections:
1. Sanity: panel counts vs raw
2. Spot-check: panel_base alignment vs features_v2
3. IV diagnostic: F-stats and correlations
4. Main DML: PLR + PLIV vs paper antigo
5. Heterogeneities + decomposition
6. Price of legislative support

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path('/Users/pedrocampelo/Library/CloudStorage/Dropbox/UnB/Doc/projects/api_camara')
PAPER_SRC = ROOT / 'paper-emendas' / 'source'
sys.path.insert(0, str(PAPER_SRC))
import _config as C
import _utils as U
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2

## 1. Panel counts (sanity)

In [ ]:
# Compare to raw and to old features_v2
raw = pd.read_csv(ROOT / 'dados/raw/votacoes/votos/votacoes_votos_raw.csv', sep=';',
                  usecols=['deputado_idLegislatura'])
v2 = pd.read_csv(ROOT / 'dados/interim/features_v2.csv', sep=';',
                  usecols=['idLegislatura'])
new = pd.read_csv(ROOT / 'dados/interim/panel/panel_base.csv', sep=';',
                   usecols=['idLegislatura'])

comp = pd.DataFrame({
    'raw': raw['deputado_idLegislatura'].value_counts().sort_index(),
    'features_v2': v2['idLegislatura'].value_counts().sort_index(),
    'panel_base (new)': new['idLegislatura'].value_counts().sort_index(),
})
comp

## 2. Alignment match: panel_base vs features_v2

In [ ]:
# Recompute alinhamento in v2 with the same logic as the old paper
v2 = pd.read_csv(ROOT/'dados/interim/features_v2.csv', sep=';',
                  usecols=['idDeputado','idVotacao','voto','d_ori_gov','d_ori_gov_liberado',
                            'd_ori_gov_sim','d_ori_gov_nao','d_ori_gov_obstrucao','d_ori_gov_abstencao'])
v2 = v2[(v2['d_ori_gov']==1) & (v2['d_ori_gov_liberado']==0)].copy()
mask_align = (
    ((v2['d_ori_gov_sim']==1) & (v2['voto']=='Sim')) |
    ((v2['d_ori_gov_nao']==1) & (v2['voto']=='Não')) |
    ((v2['d_ori_gov_abstencao']==1) & (v2['voto']=='Abstenção')) |
    ((v2['d_ori_gov_obstrucao']==1) & (v2['voto'].isin(['Obstrução','Artigo 17'])))
)
v2['alinhamento_v2'] = mask_align.astype(int)

new = pd.read_csv(ROOT/'dados/interim/panel/panel_base.csv', sep=';',
                   usecols=['idDeputado','idVotacao','alinhamento'])
merged = v2.merge(new, on=['idDeputado','idVotacao'])
match = (merged['alinhamento_v2']==merged['alinhamento']).mean()
print(f'Overlap: {len(merged):,} | match rate: {match:.4%}')

## 3. IV diagnostic

In [ ]:
fs = pd.read_csv(C.RESULTS / 'main_fstage.csv', sep=';')
fs

In [ ]:
iv = pd.read_csv(ROOT/'dados/interim/panel/iv_features.csv', sep=';')
panel = pd.read_csv(ROOT/'dados/interim/panel/panel_emendas_pre.csv', sep=';',
                     usecols=['idDeputado','idVotacao','emenda_valor'])
panel['emenda_M'] = panel['emenda_valor'] / 1e6
pf = pd.read_csv(ROOT/'dados/interim/panel/panel_features.csv', sep=';',
                  usecols=['idDeputado','idVotacao','alinhamento'])
df = panel.merge(iv, on=['idDeputado','idVotacao']).merge(pf, on=['idDeputado','idVotacao'])
for z in ['iv_fiscal_q4','iv_fiscal_pressure','iv_q4_no_ytd','iv_ytd_exec_pct']:
    rT = df[[z,'emenda_M']].corr().iloc[0,1]
    rY = df[[z,'alinhamento']].corr().iloc[0,1]
    print(f'{z:25s}  corr(Z,T)={rT:+.4f}  corr(Z,Y)={rY:+.4f}')

## 4. Main DML — comparison with paper antigo

Old paper:
- Leg 55 PLR: −0.002 (n.s.)
- Leg 55 PLIV-backlog: +0.523 ***
- Leg 56 PLR: +0.005 ***
- Leg 56 PLIV-backlog: +0.100 ***
- Pooled PLIV-backlog: +0.145 ***

These are in `coef_std` (which we now know is approximately pp/100 when T is in R\$M). Multiply by 100 for pp.

In [ ]:
old = pd.read_csv(ROOT/'dados/interim/unified_results.csv', sep=';')
old_wide = old[(old['window']=='left')].pivot_table(
    index='legis', columns=['model','iv_set'], values='coef_std')
old_wide.columns = ['_'.join(c) for c in old_wide.columns]
old_wide['note'] = 'OLD coef_std × 100 ~= pp/R$M (interpretation issue)'
old_wide

In [ ]:
new = pd.read_csv(C.RESULTS / 'main_results.csv', sep=';')
new_wide = new.pivot_table(
    index='legis', columns=['model','iv_set'], values='pp_per_unit')
new_wide.columns = ['_'.join(c) for c in new_wide.columns]
new_wide['note'] = 'NEW: pp_per_unit = pp per R$1M (correct units)'
new_wide

## 5. Heterogeneity tables

In [ ]:
for f in sorted((C.RESULTS).glob('heterogeneity_*.csv')):
    print(f'\n--- {f.name} ---')
    print(pd.read_csv(f, sep=';')[['group','model','pp_per_unit','stars','n_obs']].to_string(index=False))

## 6. Decomposition (Oaxaca-Blinder + RP9 + Polarization)

In [ ]:
for f in sorted((C.RESULTS).glob('decomp_*.csv')):
    print(f'\n--- {f.name} ---')
    print(pd.read_csv(f, sep=';').to_string(index=False))

## 7. Price of legislative support

In [ ]:
price = pd.read_csv(C.RESULTS / 'price_legislative_support.csv', sep=';')
cf = pd.read_csv(C.RESULTS / 'counterfactual_alignment.csv', sep=';')
print('Price of legislative support:'); print(price.to_string(index=False))
print('\nCounterfactual Y(T=0):'); print(cf.to_string(index=False))